## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
fatal: unable to access 'https://github.com/Lv1g1/RecSys-Challenge-2025.git/': Could not resolve host: github.com


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [3]:
import numpy as np
import pandas as pd
import gc

from Challenge.paths import load_holdout_split, XGBOOST_DATAFRAMES
from Challenge.utils import split_into_folds, train_all_models

from Challenge.features_engineering import (
    generate_candidates,
    add_models_features,
    calculate_item_item_features_fast,
    add_embedding_features,
    add_aggregate_features_stats,
    add_user_stats,
    optimize_dataframe_types,
    sanity_check
)

Running on local — storage at: /home/luigi/RecSys


## **Load Data**

In [4]:
# Random seed for reproducibility
RANDOM_SEED = 0xc0ffee

In [5]:
URM_inner, URM_outer = load_holdout_split()
URM_all = URM_inner + URM_outer
folds = split_into_folds(URM_all, n_folds=10, random_seed=RANDOM_SEED)

Fold 1/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 3043058
URM_train: 2738752
URM_validation: 304306
------------------------------
Fold 2/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 3043058
URM_train: 2738752
URM_validation: 304306
------------------------------
Fold 3/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 3043058
URM_train: 2738752
URM_validation: 304306
------------------------------
Fold 4/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 3043058
URM_train: 2738752
URM_validation: 304306
------------------------------
Fold 5/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 3043058
URM_train: 2738752
URM_validation: 304306
------------------------------
Fold 6/10
URM_all: (27095, 6969)
URM_train: (27095, 6969)
URM_validation: (27095, 6969)

URM_all: 3043058

## **Recommeder List**

In [6]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython, MatrixFactorization_BPR_Cython
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask
from implicit.cpu.als import AlternatingLeastSquares

models_mapping = {
    'TopPop': TopPop,
    'ItemKNN_cosine': ItemKNNCFRecommender,
    'ItemKNN_jaccard': ItemKNNCFRecommender,
    'ItemKNN_asymmetric': ItemKNNCFRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'ItemKNN_dice': ItemKNNCFRecommender,
    'UserKNN_cosine': UserKNNCFRecommender,
    'UserKNN_jaccard': UserKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'UserKNN_tversky': UserKNNCFRecommender,
    'UserKNN_dice': UserKNNCFRecommender,
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    # 'EASE_R': EASE_R_Recommender, Conflict with multithreading
    'P3alpha': P3alphaRecommender,
    'RP3beta': RP3betaRecommender,
    'IALS': AlternatingLeastSquares,
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
    'MatrixFactorization_BPR': MatrixFactorization_BPR_Cython,
    'NMF': NMFRecommender
}

candidate_cutoffs = {
    # Target of 10 candidates reached at Step 2 (Actual: 10.28).
    "top_10": {'UserKNN_tversky': 0, 'MultVAE': 0, 'SLIMElasticNet': 10, 'EASE_R': 5, 'IALS': 0},
    # Target of 20 candidates reached at Step 7 (Actual: 22.68).
    "top_20": {'UserKNN_tversky': 0, 'MultVAE': 5, 'SLIMElasticNet': 20, 'EASE_R': 10, 'IALS': 5},
    # Target of 30 candidates reached at Step 12 (Actual: 31.10).
    "top_30": {'UserKNN_tversky': 5, 'MultVAE': 10, 'SLIMElasticNet': 25, 'EASE_R': 20, 'IALS': 5},
    # Target of 40 candidates reached at Step 17 (Actual: 46.50).
    "top_40": {'UserKNN_tversky': 10, 'MultVAE': 15, 'SLIMElasticNet': 40, 'EASE_R': 20, 'IALS': 10},
    # Target of 50 candidates reached at Step 20 (Actual: 50.04).
    "top_50": {'UserKNN_tversky': 10, 'MultVAE': 20, 'SLIMElasticNet': 40, 'EASE_R': 25, 'IALS': 15},
    # Target of 80 candidates reached at Step 32 (Actual: 85.56).
    "top_80": {'UserKNN_tversky': 20, 'MultVAE': 40, 'SLIMElasticNet': 70, 'EASE_R': 40, 'IALS': 25}
}

## **Train Dataframe**

### **Train Models**

In [7]:
# Save slow models
slow_model_folder = "folds_10_pred"
slow_models = {'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
               'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
               'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
               'MatrixFactorization_BPR': MatrixFactorization_BPR_Cython,
               'NMF': NMFRecommender}

for fold_index, (URM_train, _) in enumerate(folds):
    print(f"Training slow models fold {fold_index+1}/{len(folds)}")

    for slow_name, class_obj in slow_models.items():
        if not os.path.exists(os.path.join(slow_model_folder, slow_name + f"_{fold_index}.zip")):
            print(f"Training slow model {slow_name} for fold {fold_index}.")

            m = train_all_models(URM_train, {slow_name: class_obj})[0][1]
            m.save_model(slow_model_folder, slow_name + f"_{fold_index}")
        
        else:
            print(f"Model {slow_name} for fold {fold_index} already exists.")

Training slow models fold 1/10
Model SLIMElasticNet for fold 0 already exists.
Model MultVAE for fold 0 already exists.
Model MatrixFactorization_WARP for fold 0 already exists.
Model MatrixFactorization_BPR for fold 0 already exists.
Model NMF for fold 0 already exists.
Training slow models fold 2/10
Model SLIMElasticNet for fold 1 already exists.
Model MultVAE for fold 1 already exists.
Model MatrixFactorization_WARP for fold 1 already exists.
Model MatrixFactorization_BPR for fold 1 already exists.
Model NMF for fold 1 already exists.
Training slow models fold 3/10
Model SLIMElasticNet for fold 2 already exists.
Model MultVAE for fold 2 already exists.
Model MatrixFactorization_WARP for fold 2 already exists.
Model MatrixFactorization_BPR for fold 2 already exists.
Model NMF for fold 2 already exists.
Training slow models fold 4/10
Model SLIMElasticNet for fold 3 already exists.
Model MultVAE for fold 3 already exists.
Model MatrixFactorization_WARP for fold 3 already exists.
Model 

In [8]:
models = []
# Train EASE before other models to avoid conflicts with multithreading
for fold_index, (URM_train, _) in enumerate(folds):
    m = train_all_models(URM_train, {'EASE_R': EASE_R_Recommender})[0][1]
    models.append([( 'EASE_R', m )])

for fold_index, (URM_train, _) in enumerate(folds):
    print(f"Training fold {fold_index+1}/{len(folds)}")

    for slow_name, class_obj in slow_models.items():
        if os.path.exists(os.path.join(slow_model_folder, slow_name + f"_{fold_index}.zip")):
            print(f"Model {slow_name} for fold {fold_index} already exists. Loading model...")

            m = class_obj(URM_train)
            m.load_model(slow_model_folder, slow_name + f"_{fold_index}")
            models[fold_index].append((slow_name, m))

            if slow_name in models_mapping:
                del models_mapping[slow_name]
        else:
            raise ValueError("  WARNING!!\nTraining slow models should have been done before!")

    models[fold_index].extend(train_all_models(URM_train, models_mapping))

  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.52 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.14 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.34 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.97 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.34 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.36 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.40 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 8.23 sec
  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
E

/home/luigi/.venvs/recsys/lib/python3.13/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/31 [00:00<?, ?it/s]

Training fold 2/10
Model SLIMElasticNet for fold 1 already exists. Loading model...
SLIMElasticNetRecommender: Loading model from file 'folds_10_predSLIMElasticNet_1'
SLIMElasticNetRecommender: Loading complete
Model MultVAE for fold 1 already exists. Loading model...
Model MatrixFactorization_WARP for fold 1 already exists. Loading model...
MatrixFactorization_WARP_Cython_Recommender: Loading model from file 'folds_10_predMatrixFactorization_WARP_1'
MatrixFactorization_WARP_Cython_Recommender: Loading complete
Model MatrixFactorization_BPR for fold 1 already exists. Loading model...
MatrixFactorization_BPR_Cython_Recommender: Loading model from file 'folds_10_predMatrixFactorization_BPR_1'
MatrixFactorization_BPR_Cython_Recommender: Loading complete
Model NMF for fold 1 already exists. Loading model...
NMFRecommender: Loading model from file 'folds_10_predNMF_1'
NMFRecommender: Loading complete
  Training model: TopPop
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%), 

  0%|          | 0/31 [00:00<?, ?it/s]

...

  0%|          | 0/31 [00:00<?, ?it/s]

Training fold 9/10
Model SLIMElasticNet for fold 8 already exists. Loading model...
SLIMElasticNetRecommender: Loading model from file 'folds_10_predSLIMElasticNet_8'
SLIMElasticNetRecommender: Loading complete
Model MultVAE for fold 8 already exists. Loading model...
Model MatrixFactorization_WARP for fold 8 already exists. Loading model...
MatrixFactorization_WARP_Cython_Recommender: Loading model from file 'folds_10_predMatrixFactorization_WARP_8'
MatrixFactorization_WARP_Cython_Recommender: Loading complete
Model MatrixFactorization_BPR for fold 8 already exists. Loading model...
MatrixFactorization_BPR_Cython_Recommender: Loading model from file 'folds_10_predMatrixFactorization_BPR_8'
MatrixFactorization_BPR_Cython_Recommender: Loading complete
Model NMF for fold 8 already exists. Loading model...
NMFRecommender: Loading model from file 'folds_10_predNMF_8'
NMFRecommender: Loading complete
  Training model: TopPop
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%), 

  0%|          | 0/31 [00:00<?, ?it/s]

Training fold 10/10
Model SLIMElasticNet for fold 9 already exists. Loading model...
SLIMElasticNetRecommender: Loading model from file 'folds_10_predSLIMElasticNet_9'
SLIMElasticNetRecommender: Loading complete
Model MultVAE for fold 9 already exists. Loading model...
Model MatrixFactorization_WARP for fold 9 already exists. Loading model...
MatrixFactorization_WARP_Cython_Recommender: Loading model from file 'folds_10_predMatrixFactorization_WARP_9'
MatrixFactorization_WARP_Cython_Recommender: Loading complete
Model MatrixFactorization_BPR for fold 9 already exists. Loading model...
MatrixFactorization_BPR_Cython_Recommender: Loading model from file 'folds_10_predMatrixFactorization_BPR_9'
MatrixFactorization_BPR_Cython_Recommender: Loading complete
Model NMF for fold 9 already exists. Loading model...
NMFRecommender: Loading model from file 'folds_10_predNMF_9'
NMFRecommender: Loading complete
  Training model: TopPop
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%),

  0%|          | 0/31 [00:00<?, ?it/s]

In [9]:
models

[[('EASE_R',
   <Recommenders.EASE_R.EASE_R_Recommender.EASE_R_Recommender at 0x7f32b2e11940>),
  ('SLIMElasticNet',
   <Recommenders.SLIM.SLIMElasticNetRecommender.MultiThreadSLIM_SLIMElasticNetRecommender at 0x7f32b2e11a90>),
  ('MultVAE',
   <Recommenders.Neural.MultVAE_PyTorch_Recommender.MultVAERecommender_PyTorch_OptimizerMask at 0x7f32b2e11be0>),
  ('MatrixFactorization_WARP',
   <Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython.MatrixFactorization_WARP_Cython at 0x7f32b2e12660>),
  ('MatrixFactorization_BPR',
   <Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython.MatrixFactorization_BPR_Cython at 0x7f32b2e127b0>),
  ('NMF',
   <Recommenders.MatrixFactorization.NMFRecommender.NMFRecommender at 0x7f32b2e12900>),
  ('TopPop',
   <Recommenders.NonPersonalizedRecommender.TopPop at 0x7f32b2e12ba0>),
  ('ItemKNN_cosine',
   <Recommenders.KNN.ItemKNNCFRecommender.ItemKNNCFRecommender at 0x7f32b2e12cf0>),
  ('ItemKNN_jaccard',
   <Recommenders.KNN.ItemK

### **Generate Dataframe**

In [14]:
def run_oof_pipeline(folds, models, models_cutoff, n_overlap=2, save_path=".") -> pd.DataFrame:
    """
    Main loop to generate OOF data with configurable overlap.
    
    Args:
        folds: List of (URM_train, URM_val) tuples
        models: List of models per fold
        models_cutoff: Dict of cutoffs
        n_overlap (int): Number of folds each user should be processed in.
                         1 = Standard OOF (each user in 1 fold).
                         2 = Double Diagonal (your previous strategy).
                         N_FOLDS = Full saturation (each user in every fold).
        save_path (str): Directory to save fold data.
    """
    os.makedirs(save_path, exist_ok=True)
    N_FOLDS = len(folds)
    
    # Get total users from first fold
    n_users = folds[0][0].shape[0]
    all_indices = np.arange(n_users)

    # Pre-calculate user remainders to speed up loop
    # This maps every user to their "base fold" (0 to 9)
    user_base_folds = all_indices % N_FOLDS
    
    for fold_idx, (URM_train, URM_val) in enumerate(folds):
        print(f"\n==================================================")
        print(f"PROCESSING FOLD {fold_idx+1}/{len(folds)}")
        print(f"==================================================")

        # ---------------------------------------------------------
        # GENERALIZED USER ASSIGNMENT
        # ---------------------------------------------------------
        # We need to find which 'base folds' are active for this current fold_idx.
        # User u (base b) is processed in Fold F if F is in [b, b+1, ..., b+n_overlap-1]
        # Inverting this: In Fold F, we accept base b if b = (F - offset) % N
        
        # Calculate the list of "base assignments" that target this fold
        active_bases = [(fold_idx - i) % N_FOLDS for i in range(n_overlap)]

        # Select users who belong to these base assignments
        mask = np.isin(user_base_folds, active_bases)
        target_users = all_indices[mask]
        
        print(f"Overlap: {n_overlap} | Active Base Groups: {active_bases}")
        print(f"Target Users for this fold: {len(target_users)} (approx {n_users * n_overlap / N_FOLDS:.0f})")
        
        if len(target_users) == 0:
            continue
        
        models_list = models[fold_idx]

        # Generate Candidates
        print(f"--- Generating Candidates ---")
        df_fold = generate_candidates(
            URM_train[target_users],
            user_ids=target_users,
            models=models_list,
            models_cutoff=models_cutoff
        )
        print(f"Candidates: {len(df_fold)}")

        # Add Fold column
        df_fold['Fold'] = fold_idx

        # Add Labels
        print(f"--- Adding Labels ---")
        gt_values = URM_val[df_fold['UserID'].values, df_fold['ItemID'].values]
        df_fold['Label'] = (np.array(gt_values).squeeze() > 0).astype(np.int8)
        
        positives = df_fold['Label'].sum()
        print(f"Positives found: {positives} (Ratio: {positives/len(df_fold):.4f})")

        # Add Features
        print(f"--- Adding Features ---")
        
        # 1. Model Scores & Ranks
        df_fold = add_models_features(df_fold, URM_train, models_list)
        
        # 2. Item-Item Similarity (SLIM Only)
        # Filter models list to find SLIM
        sim_models = [m for m in models_list if
                       'SLIM' in m[0] or
                       'RP3beta' in m[0] or
                       'ItemKNN_tversky' in m[0]]
        if sim_models:
            df_fold = calculate_item_item_features_fast(df_fold, URM_train, sim_models)
            
        # 3. Embedding Features (IALS Only)
        # Filter to find IALS
        ials_model = next((m[1] for m in models_list if 'IALS' in m[0]), None)
        if ials_model:
            df_fold = add_embedding_features(df_fold, ials_model)
            
        # 4. Aggregates (Mean/Std/Min/Max)
        df_fold = add_aggregate_features_stats(df_fold)
        
        # 5. User Stats (Mainstreamness)
        df_fold = add_user_stats(df_fold, URM_train)

        # Optimize Memory
        print(f"--- Optimizing Types ---")
        df_fold = optimize_dataframe_types(df_fold)
        
        print(f"Final Inference Shape: {df_fold.shape}")
        print(f"Memory Usage: {df_fold.memory_usage().sum() / 1e6:.2f} MB")

        # Check
        assert sanity_check(df_fold), f"Sanity check failed for fold {fold_idx}"

        # Save
        save_path_fold = os.path.join(save_path, f"prediction_train_OOF_fold{fold_idx}.parquet")
        df_fold.to_parquet(
            path=save_path_fold,
            engine='fastparquet',
            compression='snappy',
            index=False
        )
        
        # Cleanup to save RAM
        del df_fold, models_list, URM_train, URM_val
        gc.collect()

In [18]:
save_path = os.path.join(XGBOOST_DATAFRAMES, "OOF_folds")
run_oof_pipeline(folds, models, candidate_cutoffs, n_overlap=10, save_path=save_path)


PROCESSING FOLD 1/10
Overlap: 10 | Active Base Groups: [0, 9, 8, 7, 6, 5, 4, 3, 2, 1]
Target Users for this fold: 27095 (approx 27095)
--- Generating Candidates ---
Computing candidates with EASE_R...
Computing candidates with EASE_R (Max Cutoff: 40)...
Computing candidates with SLIMElasticNet...
Computing candidates with SLIMElasticNet (Max Cutoff: 70)...
Computing candidates with MultVAE...
Computing candidates with MultVAE (Max Cutoff: 40)...
Computing candidates with MatrixFactorization_WARP...
Skipping MatrixFactorization_WARP (Max cutoff is 0)...
Computing candidates with MatrixFactorization_BPR...
Skipping MatrixFactorization_BPR (Max cutoff is 0)...
Computing candidates with NMF...
Skipping NMF (Max cutoff is 0)...
Computing candidates with TopPop...
Skipping TopPop (Max cutoff is 0)...
Computing candidates with ItemKNN_cosine...
Skipping ItemKNN_cosine (Max cutoff is 0)...
Computing candidates with ItemKNN_jaccard...
Skipping ItemKNN_jaccard (Max cutoff is 0)...
Computing can

100%|██████████| 27095/27095 [00:09<00:00, 2794.20it/s]


Extracting MAX similarity for ItemKNN_tversky...


100%|██████████| 27095/27095 [00:06<00:00, 4059.84it/s]


Extracting MAX similarity for RP3beta...


...

Assigning columns to DataFrame...
  Calculating User Eccentricity...
Calculating Rank Stats on 20 models...


/home/luigi/RecSys/Challenge/features_engineering.py:536: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Mean_RankPosition'] = df[position_columns].mean(axis=1)
/home/luigi/RecSys/Challenge/features_engineering.py:537: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Std_RankPosition'] = df[position_columns].std(axis=1)
/home/luigi/RecSys/Challenge/features_engineering.py:540: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Con

Calculating Vote Count on 20 models...
Calculating Score Stats & Ratios on 21 models...


/home/luigi/RecSys/Challenge/features_engineering.py:562: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Mean_Score'] = df[score_columns].mean(axis=1)
/home/luigi/RecSys/Challenge/features_engineering.py:563: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Std_Score'] = df[score_columns].std(axis=1)
/home/luigi/RecSys/Challenge/features_engineering.py:564: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all co

--- Optimizing Types ---
Optimizing memory usage...
Dropping constant columns: ['top_80', 'Fold']
Final Inference Shape: (2283195, 158)
Memory Usage: 1093.65 MB
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

--- SANITY CHECK PASSED: Data is clean ---


## **Prediction Dataframe**

### **Train Models**

In [7]:
m = train_all_models(URM_all, {'EASE_R': EASE_R_Recommender})[0][1]
models = [( 'EASE_R', m )]
models.extend(train_all_models(URM_all, models_mapping))

  Training model: EASE_R
EASE_R_Recommender: Fitting model... 
EASE_R_Recommender: Fitting model... done in 7.24 sec
  Training model: TopPop
  Training model: ItemKNN_cosine
Similarity column 6969 (100.0%), 3580.49 column/sec. Elapsed time 1.95 sec
  Training model: ItemKNN_jaccard
Similarity column 6969 (100.0%), 3536.85 column/sec. Elapsed time 1.97 sec
  Training model: ItemKNN_asymmetric
Similarity column 6969 (100.0%), 3470.76 column/sec. Elapsed time 2.01 sec
  Training model: ItemKNN_tversky
Similarity column 6969 (100.0%), 3510.37 column/sec. Elapsed time 1.99 sec
  Training model: ItemKNN_dice
Similarity column 6969 (100.0%), 3558.83 column/sec. Elapsed time 1.96 sec
  Training model: UserKNN_cosine
Similarity column 27095 (100.0%), 1454.84 column/sec. Elapsed time 18.62 sec
  Training model: UserKNN_jaccard
Similarity column 27095 (100.0%), 1452.94 column/sec. Elapsed time 18.65 sec
  Training model: UserKNN_asymmetric
Similarity column 27095 (100.0%), 1465.78 column/sec. El

100%|█████████▉| 6968/6969 [02:09<00:00, 53.73it/s]


  Training model: P3alpha
P3alphaRecommender: Similarity column 6969 (100.0%), 2712.98 column/sec. Elapsed time 2.57 sec
  Training model: RP3beta
RP3betaRecommender: Similarity column 6969 (100.0%), 3141.80 column/sec. Elapsed time 2.22 sec
  Training model: IALS


/home/luigi/.venvs/recsys/lib/python3.13/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/31 [00:00<?, ?it/s]

  Training model: MatrixFactorization_WARP
MF_WARP: Processed 27648 (100.0%) in 0.92 sec. MSE loss 8.08E-02. Sample per second: 29919
MF_WARP: Epoch 1 of 1500. Elapsed time 0.16 sec
MF_WARP: Processed 27648 (100.0%) in 1.07 sec. MSE loss 1.41E-01. Sample per second: 25852
MF_WARP: Epoch 2 of 1500. Elapsed time 0.30 sec
MF_WARP: Processed 27648 (100.0%) in 0.21 sec. MSE loss 1.81E-01. Sample per second: 129702
MF_WARP: Epoch 3 of 1500. Elapsed time 0.45 sec
MF_WARP: Processed 27648 (100.0%) in 0.36 sec. MSE loss 2.22E-01. Sample per second: 76894
MF_WARP: Epoch 4 of 1500. Elapsed time 0.59 sec
MF_WARP: Processed 27648 (100.0%) in 0.50 sec. MSE loss 2.60E-01. Sample per second: 54839
MF_WARP: Epoch 5 of 1500. Elapsed time 0.74 sec
MF_WARP: Processed 27648 (100.0%) in 0.65 sec. MSE loss 3.17E-01. Sample per second: 42446
MF_WARP: Epoch 6 of 1500. Elapsed time 0.88 sec
MF_WARP: Processed 27648 (100.0%) in 0.80 sec. MSE loss 4.31E-01. Sample per second: 34653
MF_WARP: Epoch 7 of 1500. Elaps

/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:330: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(user_batch_tensor.indptr,


MultVAERecommender_PyTorch: Epoch 1 of 70. Elapsed time 1.12 sec
MultVAERecommender_PyTorch: Epoch 2 of 70. Elapsed time 1.97 sec
MultVAERecommender_PyTorch: Epoch 3 of 70. Elapsed time 2.83 sec
MultVAERecommender_PyTorch: Epoch 4 of 70. Elapsed time 3.69 sec
MultVAERecommender_PyTorch: Epoch 5 of 70. Elapsed time 4.55 sec
MultVAERecommender_PyTorch: Epoch 6 of 70. Elapsed time 5.41 sec
MultVAERecommender_PyTorch: Epoch 7 of 70. Elapsed time 6.27 sec
MultVAERecommender_PyTorch: Epoch 8 of 70. Elapsed time 7.12 sec
MultVAERecommender_PyTorch: Epoch 9 of 70. Elapsed time 7.97 sec
MultVAERecommender_PyTorch: Epoch 10 of 70. Elapsed time 8.82 sec
MultVAERecommender_PyTorch: Epoch 11 of 70. Elapsed time 9.67 sec
MultVAERecommender_PyTorch: Epoch 12 of 70. Elapsed time 10.53 sec
MultVAERecommender_PyTorch: Epoch 13 of 70. Elapsed time 11.38 sec
MultVAERecommender_PyTorch: Epoch 14 of 70. Elapsed time 12.23 sec
MultVAERecommender_PyTorch: Epoch 15 of 70. Elapsed time 13.08 sec
MultVAERecommen

### **Generate Dataframe**

In [8]:
def generate_inference_data(
        URM_train, 
        target_users, 
        models, 
        models_cutoff):
    """
    Generates the dataframe for Validation or Test.
    
    Args:
        URM_train: The URM used to train the models (e.g. URM_inner).
                   Features like 'Similarity to History' are calculated against this.
        target_users: Array of users to predict for.
        models: List of tuples [('SLIM', model), ...] (Trained on URM_train).
        models_cutoff: The cutoff dictionary.
    """
    
    print(f"Generating inference data for {len(target_users)} users...")
    
    # 1. Generate Candidates
    print(f"--- Generating Candidates ---")
    df = generate_candidates(
        URM_train[target_users],
        user_ids=target_users,
        models=models,
        models_cutoff=models_cutoff
    )
    print(f"Candidates: {len(df)}")
    
    # 2. Add Model Features (Scores & Ranks)
    print(f"--- Adding Model Features ---")
    df = add_models_features(df, URM_train, models)
    
    # 3. Add Item-Item Similarity Features
    sim_models = [m for m in models if
                    'SLIM' in m[0] or
                    'RP3beta' in m[0] or
                    'ItemKNN_tversky' in m[0]]
    if sim_models:
        print(f"--- Adding SLIM Features ---")
        df = calculate_item_item_features_fast(df, URM_train, sim_models)
        
    # 4. Add Embedding Features (IALS Only)
    ials_model = next((m[1] for m in models if 'IALS' in m[0]), None)
    if ials_model:
        print(f"--- Adding IALS Embeddings ---")
        df = add_embedding_features(df, ials_model)
        
    # 5. Add Aggregate Features
    print(f"--- Adding Aggregates ---")
    df = add_aggregate_features_stats(df)
    
    # 6. Add User Stats (Mainstreamness)
    print(f"--- Adding User Stats ---")
    df = add_user_stats(df, URM_train)
    
    # 7. Optimize Memory
    print(f"--- Optimizing Types ---")
    df = optimize_dataframe_types(df)
    
    # Check
    assert sanity_check(df), f"Sanity check failed for inference data"

    print(f"Final Inference Shape: {df.shape}")
    print(f"Memory Usage: {df.memory_usage().sum() / 1e6:.2f} MB")

    return df

In [9]:
target_users = np.arange(URM_all.shape[0])
df_val = generate_inference_data(URM_all, target_users, models, candidate_cutoffs)

Generating inference data for 27095 users...
--- Generating Candidates ---
Computing candidates with EASE_R...
Computing candidates with EASE_R (Max Cutoff: 40)...
Computing candidates with TopPop...
Skipping TopPop (Max cutoff is 0)...
Computing candidates with ItemKNN_cosine...
Skipping ItemKNN_cosine (Max cutoff is 0)...
Computing candidates with ItemKNN_jaccard...
Skipping ItemKNN_jaccard (Max cutoff is 0)...
Computing candidates with ItemKNN_asymmetric...
Skipping ItemKNN_asymmetric (Max cutoff is 0)...
Computing candidates with ItemKNN_tversky...
Skipping ItemKNN_tversky (Max cutoff is 0)...
Computing candidates with ItemKNN_dice...
Skipping ItemKNN_dice (Max cutoff is 0)...
Computing candidates with UserKNN_cosine...
Skipping UserKNN_cosine (Max cutoff is 0)...
Computing candidates with UserKNN_jaccard...
Skipping UserKNN_jaccard (Max cutoff is 0)...
Computing candidates with UserKNN_asymmetric...
Skipping UserKNN_asymmetric (Max cutoff is 0)...
Computing candidates with UserKNN

100%|██████████| 27095/27095 [00:05<00:00, 4586.96it/s]


Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:08<00:00, 3156.68it/s]


Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:06<00:00, 4387.58it/s]


--- Adding IALS Embeddings ---
  Running PCA to extract top 5 components...
  Calculating Vector Norms...
Clustering Users and Items...
Computing features in batches of 500...


100%|██████████| 4570/4570 [00:00<00:00, 6147.47it/s]


Assigning columns to DataFrame...
  Calculating User Eccentricity...
--- Adding Aggregates ---
Calculating Rank Stats on 20 models...
Calculating Vote Count on 20 models...
Calculating Score Stats & Ratios on 21 models...
--- Adding User Stats ---
--- Optimizing Types ---
Optimizing memory usage...
Dropping constant columns: ['top_80']
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

--- SANITY CHECK PASSED: Data is clean ---
Final Inference Shape: (2284689, 157)
Memory Usage: 1092.08 MB


### **Save Dataframe**

In [10]:
# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "prediction_OOF.parquet")
os.makedirs(os.path.dirname(save_path), exist_ok=True)

df_val.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='snappy',
    index=False
)